# PATSTAT uniqueC Trend — yearly de-duplicated incoming citations per application

The twin of `PatentView/notebook/patent_uniqueC_trend.ipynb`: **`uniqueC` by year since filing**, the
trajectory of the de-duplicated count that `patstat_citation.parquet` gives only at four windows.

## Input
```
PATSTAT/output/patstat_reference.parquet       # the edges (ps.EDGE_WHERE: age >= 0, not replenished)
PATSTAT/output/patstat_citation.parquet        # uniqueC_{3,5,10,all}: the window check
PATSTAT/output/patstat_citation_trend.parquet  # uniqueC per (application, citing year): the identity check
```

## Definition
For each (cited, citing) pair the **earliest** lag over its edges, `d = min(age)`; a citing application
is counted once, in the year of its earliest citation: `uniqueC(t) = #{citing : d = t}`, so
`Σ_{t ≤ w} uniqueC(t) = uniqueC_w`.

**In PATSTAT this file equals the `uniqueC` column of `patstat_citation_trend`.** Both ends are dated by
the *application's* filing year, so every edge between one citing and one cited application has the same
age and the "earliest" lag is the only lag. (In PatentView it is not redundant: grant-year clocks of the
granted and pre-grant records of one citing patent differ.) The file is kept so code written against
`patent_uniqueC_trend` runs unchanged on PATSTAT; section 2 asserts the identity.

## Output
`PATSTAT/output/patstat_uniqueC_trend.parquet` — `appln_id, filing_year, cite_year, yrs_since_filing, uniqueC`
(one row per application × year with ≥ 1 new citing application).

> **Clock switch.** This text describes the default filing clock (`output/`). With `NB_PS_CLOCK=grant` the same notebook runs on the grant clock, PatentView's: years and ages are grant years (first publication with `publn_first_grant = 'Y'`), only applications with a grant year are in the universe (both ends of every edge), year / age columns are named `grant_year` / `yrs_since_grant`, and the outputs go to `PATSTAT/output_grant/` (executed copy: `notebook/grant/`).

> **Family unit.** With `NB_PS_UNIT=family` the same notebook runs with the DOCDB family as the unit (`docdb_family_id`), dated by the family's earliest priority year (`priority_year`, `yrs_since_priority`): citations are distinct family-to-family pairs with within-family citations dropped, classification comes from the earliest member that has one, inventors and their countries from one representative member (the one with the most located inventors), and the outputs go to `PATSTAT/output_family/` (executed copy: `notebook/family/`).


In [1]:
import os, sys, gc, time
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/PATSTAT')
import ps_common as ps
OUT = ps.OUT
REF, CIT, TRD = ps.out('patstat_reference.parquet'), ps.out('patstat_citation.parquet'), ps.out('patstat_citation_trend.parquet')
OUT_FP = ps.out('patstat_uniqueC_trend.parquet')
ps.preflight('patstat_uniqueC_trend')
con = ps.connect()
print('edge filter:', ps.EDGE_WHERE)

dump   : /project/jevans/PATSTAT/unzipped_data   (65 zip parts, 11 tables registered)
raw    : /project/jevans/Dawoon/Science of Science/PATSTAT/raw
output : /project/jevans/Dawoon/Science of Science/PATSTAT/output

  patstat_uniqueC_trend         OK
edge filter: age >= 0 AND NOT replenished


## 1. Build

In [2]:
%%time
t0 = time.time()
con.execute(f"""COPY (
  WITH e AS (SELECT cited_id, citing_id, min(age) AS d, any_value(cited_year) AS fy
             FROM read_parquet('{REF}') WHERE {ps.EDGE_WHERE} GROUP BY 1, 2)
  SELECT cited_id AS {ps.KEY}, fy::INTEGER AS {ps.YEAR_COL}, (fy + d)::INTEGER AS cite_year,
         d::INTEGER AS {ps.AGE_COL}, count(*)::BIGINT AS uniqueC
  FROM e GROUP BY 1, 2, 3, 4 ORDER BY 1, 4
) TO '{OUT_FP}.tmp' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 1000000)""")
os.replace(OUT_FP + '.tmp', OUT_FP)
st = con.execute(f"""SELECT count(*) AS rows, count(DISTINCT {ps.KEY}) AS documents, sum(uniqueC) AS citing_pairs,
                      min(cite_year) AS y0, max(cite_year) AS y1, max({ps.AGE_COL}) AS max_age FROM read_parquet('{OUT_FP}')""").df().iloc[0]
print(f'[{time.time() - t0:.0f}s] WROTE {OUT_FP}  ({os.path.getsize(OUT_FP) / 1e6:.0f} MB)')
print('  ' + '  '.join(f'{k} {int(v):,}' for k, v in st.items()))

[26s] WROTE /project/jevans/Dawoon/Science of Science/PATSTAT/output/patstat_uniqueC_trend.parquet  (368 MB)
  rows 157,842,642  applications 40,901,611  citing_pairs 335,409,894  y0 1,900  y1 2,023  max_age 123


## 2. Checks — window totals, and the identity with `patstat_citation_trend.uniqueC`

In [3]:
%%time
chk = con.execute(f"""
WITH s AS (
  SELECT {ps.KEY},
         sum(uniqueC) FILTER (WHERE {ps.AGE_COL} <= 3)  AS u3, sum(uniqueC) FILTER (WHERE {ps.AGE_COL} <= 5) AS u5,
         sum(uniqueC) FILTER (WHERE {ps.AGE_COL} <= 10) AS u10, sum(uniqueC) AS uall
  FROM read_parquet('{OUT_FP}') GROUP BY 1),
  j AS (SELECT c.{ps.KEY}, c.uniqueC_3, c.uniqueC_5, c.uniqueC_10, c.uniqueC_all,
               coalesce(s.u3, 0) AS u3, coalesce(s.u5, 0) AS u5, coalesce(s.u10, 0) AS u10, coalesce(s.uall, 0) AS uall
        FROM read_parquet('{CIT}') c FULL JOIN s USING ({ps.KEY}))
SELECT count(*) AS applications,
       count(*) FILTER (WHERE uniqueC_3 IS DISTINCT FROM u3)   AS bad_3,
       count(*) FILTER (WHERE uniqueC_5 IS DISTINCT FROM u5)   AS bad_5,
       count(*) FILTER (WHERE uniqueC_10 IS DISTINCT FROM u10) AS bad_10,
       count(*) FILTER (WHERE uniqueC_all IS DISTINCT FROM uall) AS bad_all
FROM j""").df().iloc[0]
for k, v in chk.items():
    print(f'  {k:<16}{int(v):>15,}')
idn = con.execute(f"""
SELECT count(*) AS rows_compared,
       count(*) FILTER (WHERE u.uniqueC IS DISTINCT FROM t.uniqueC) AS identity_mismatch
FROM read_parquet('{OUT_FP}') u FULL JOIN read_parquet('{TRD}') t
  ON u.{ps.KEY} = t.{ps.KEY} AND u.{ps.AGE_COL} = t.{ps.AGE_COL}""").df().iloc[0]
print(f"  vs patstat_citation_trend.uniqueC: {int(idn.rows_compared):,} (application, year) rows, "
      f"{int(idn.identity_mismatch):,} differ")
ok = all(chk[f'bad_{w}'] == 0 for w in ('3', '5', '10', 'all')) and idn.identity_mismatch == 0
print('\nyearly series reproduces uniqueC_3/5/10/all and equals citation_trend.uniqueC: ' + ('YES' if ok else 'NO'))
display(con.execute(f"SELECT * FROM read_parquet('{OUT_FP}') ORDER BY uniqueC DESC LIMIT 8").df())
con.close()
assert ok

  applications         40,901,611
  bad_3                         0
  bad_5                         0
  bad_10                        0
  bad_all                       0
  vs patstat_citation_trend.uniqueC: 157,842,642 (application, year) rows, 0 differ

yearly series reproduces uniqueC_3/5/10/all and equals citation_trend.uniqueC: YES


,appln_id,filing_year,cite_year,yrs_since_filing,uniqueC
0,49655670,1994,2002,8,1052
1,51239601,1983,2017,34,666
2,378600018,2011,2017,6,656
3,51239601,1983,2016,33,636
4,51239601,1983,2015,32,634
5,421943759,2013,2017,4,628
6,49790511,2006,2017,11,625
7,51239601,1983,2010,27,623
